In [39]:

!mkdir dysarthric-asr && cd dysarthric-asr
!mkdir data notebooks scripts

In [40]:
!pip install pandas numpy librosa soundfile jupyterlab kaggle

In [41]:
!cd data
!wget https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2
!tar -xjvf LJSpeech-1.1.tar.bz2
!rm LJSpeech-1.1.tar.bz2

--2026-05-28 11:05:43--  https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2
Resolving data.keithito.com (data.keithito.com)... 169.150.249.162, 2400:52e0:1a01::1109:1
Connecting to data.keithito.com (data.keithito.com)|169.150.249.162|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2748572632 (2.6G) [text/plain]
Saving to: ‘LJSpeech-1.1.tar.bz2’

LJSpeech-1.1.tar.bz 100%[===================>]   2.56G   126MB/s    in 22s     

2026-05-28 11:06:05 (120 MB/s) - ‘LJSpeech-1.1.tar.bz2’ saved [2748572632/2748572632]

LJSpeech-1.1/
LJSpeech-1.1/metadata.csv
LJSpeech-1.1/wavs/
LJSpeech-1.1/wavs/LJ007-0048.wav
LJSpeech-1.1/wavs/LJ041-0060.wav
LJSpeech-1.1/wavs/LJ005-0133.wav
LJSpeech-1.1/wavs/LJ039-0220.wav
LJSpeech-1.1/wavs/LJ048-0274.wav
LJSpeech-1.1/wavs/LJ036-0043.wav
LJSpeech-1.1/wavs/LJ037-0129.wav
LJSpeech-1.1/wavs/LJ011-0265.wav
LJSpeech-1.1/wavs/LJ028-0418.wav
LJSpeech-1.1/wavs/LJ042-0116.wav
LJSpeech-1.1/wavs/LJ005-0143.wav
LJSpeech-1.1/wavs/LJ035-008

In [42]:
!mkdir -p ~/.kaggle && echo KGAT_31e36cbb338d65f1cbdb1fa54ff4f57b > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

In [43]:
mkdir dysarthic-asr

In [44]:
cd dysarthic-asr/

/root/dysarthic-asr


In [45]:
!kaggle datasets download -d iamhungundji/torgo-dataset

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata


In [46]:
!kaggle datasets download -d pranaykoppula/torgo-audio

Dataset URL: https://www.kaggle.com/datasets/pranaykoppula/torgo-audio
License(s): other
 28% 363M/1.29G [00:05<00:13, 72.7MB/s]
User cancelled operation


In [47]:
!unzip -q torgo-audio.zip -d TORGO

[torgo-audio.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of torgo-audio.zip or
        torgo-audio.zip.zip, and cannot find torgo-audio.zip.ZIP, period.


In [48]:
!rm torgo-audio.zip

In [49]:
!rm LJSpeech-1.1.tar.bz2

rm: cannot remove 'LJSpeech-1.1.tar.bz2': No such file or directory


In [50]:
cd

/root


In [51]:
import os
import pandas as pd
from pathlib import Path

# Setup paths relative to your notebook
lj_path = Path('/content/data/LJSpeech-1.1')
torgo_path = Path('/content/data/dysarthic-asr/TORGO')
master_data = []

# ==========================================
# 1. Parse LJ Speech
# ==========================================
print("Parsing LJ Speech...")
lj_meta_file = lj_path / 'metadata.csv'
if lj_meta_file.exists():
    # LJ Speech uses pipe-separated values without a header
    df_lj = pd.read_csv(lj_meta_file, sep='|', header=None, names=['ID', 'transcript', 'normalized_transcript'])

    for index, row in df_lj.iterrows():
        # Construct the path to the wav file
        wav_file = lj_path / 'wavs' / f"{row['ID']}.wav"

        # We only add it if the file actually exists
        if wav_file.exists():
             master_data.append({
                'audio_filepath': str(wav_file.resolve()), # Use absolute paths to be safe
                'transcript': row['normalized_transcript'],
                'speaker_id': 'LJ',
                'dataset': 'LJSpeech',
                'is_dysarthric': False
            })
else:
    print(f"Warning: LJ Speech metadata not found at {lj_meta_file}")

# ==========================================
# 2. Parse TORGO (Using the pranaykoppula structure)
# ==========================================
print("Parsing TORGO...")

# The structure is usually split into Female/Male and Control/Dysarthric folders
torgo_categories = ['F_Con', 'F_Dys', 'M_Con', 'M_Dys']

for category in torgo_categories:
    cat_path = torgo_path / category
    if not cat_path.exists():
        continue

    is_dys = category.endswith('_Dys')

    # Iterate through each speaker in the category (e.g., F01, FC01)
    for speaker_folder in os.listdir(cat_path):
        speaker_path = cat_path / speaker_folder
        if not speaker_path.is_dir(): continue

        # Inside a speaker folder, the layout might vary slightly, but we need to find
        # paired .wav and .txt files. A common pattern in this mirror is having
        # 'wav' and 'prompts' directories, or having them flat.
        # Let's do a recursive search to be safe.

        for root, _, files in os.walk(speaker_path):
            for file in files:
                if file.endswith('.wav'):
                    wav_full_path = Path(root) / file

                    # Assuming the prompt file has the same name but ends in .txt
                    # We need to find where it lives.
                    txt_name = file.replace('.wav', '.txt')

                    # Look for the prompt file in the same directory, or a parallel 'prompts' dir
                    possible_txt_paths = [
                        wav_full_path.with_suffix('.txt'),
                        Path(root).parent / 'prompts' / txt_name
                    ]

                    transcript = None
                    for txt_path in possible_txt_paths:
                        if txt_path.exists():
                            try:
                                with open(txt_path, 'r', encoding='utf-8') as f:
                                    transcript = f.read().strip()
                                break # Found it
                            except UnicodeDecodeError:
                                # Sometimes files have weird encodings
                                pass

                    if transcript:
                        master_data.append({
                            'audio_filepath': str(wav_full_path.resolve()),
                            'transcript': transcript,
                            'speaker_id': speaker_folder,
                            'dataset': 'TORGO',
                            'is_dysarthric': is_dys
                        })

# ==========================================
# 3. Compile and Save
# ==========================================
df_master = pd.DataFrame(master_data)

# Drop any rows where the transcript might be null (LJ Speech sometimes has empty rows)
df_master.dropna(subset=['transcript'], inplace=True)

# Save to CSV
output_path = Path('/content/data/master_index.csv')
df_master.to_csv(output_path, index=False)

print(f"Index successfully built! Total samples: {len(df_master)}")
print(f"Saved to: {output_path.resolve()}")
df_master.head()

Parsing LJ Speech...
Parsing TORGO...
Index successfully built! Total samples: 13084
Saved to: /content/data/master_index.csv


,audio_filepath,transcript,speaker_id,dataset,is_dysarthric
0,/content/data/LJSpeech-1.1/wavs/LJ001-0001.wav,"Printing, in the only sense with which we are ...",LJ,LJSpeech,False
1,/content/data/LJSpeech-1.1/wavs/LJ001-0002.wav,in being comparatively modern.,LJ,LJSpeech,False
2,/content/data/LJSpeech-1.1/wavs/LJ001-0003.wav,For although the Chinese took impressions from...,LJ,LJSpeech,False
3,/content/data/LJSpeech-1.1/wavs/LJ001-0004.wav,"produced the block books, which were the immed...",LJ,LJSpeech,False
4,/content/data/LJSpeech-1.1/wavs/LJ001-0005.wav,the invention of movable metal letters in the ...,LJ,LJSpeech,False


In [52]:
import pandas as pd
import re
from pathlib import Path

# Load the index we built in the last step
index_path = Path('/content/data/master_index.csv')
df = pd.read_csv(index_path)

def clean_transcript(text):
    if not isinstance(text, str):
        return ""

    # 1. Remove all text inside brackets (e.g., [breath], [smack], [noise])
    text = re.sub(r'\[.*?\]', '', text)

    # NEW: Replace hyphens with spaces to separate hyphenated words
    text = text.replace('-', ' ')

    # 2. Remove asterisks, punctuation, and special characters (keep alphanumeric and spaces)
    text = re.sub(r'[^a-zA-Z0-9\s\']', '', text)

    # 3. Convert to uppercase
    text = text.upper()

    # 4. Remove extra whitespace caused by the deletions
    text = re.sub(r'\s+', ' ', text).strip()

    return text
print("Cleaning transcripts...")
df['clean_transcript'] = df['transcript'].apply(clean_transcript)

# Drop rows that became completely empty after cleaning (e.g., a file that was just "[breath]")
df = df[df['clean_transcript'] != ""]

# Save the cleaned index
clean_index_path = Path('/content/data/master_index_clean.csv')
df.to_csv(clean_index_path, index=False)

print(f"Cleanup complete! Saved to {clean_index_path.resolve()}")

# Let's look at a before-and-after comparison
df[['transcript', 'clean_transcript']].sample(10)

Cleaning transcripts...
Cleanup complete! Saved to /content/data/master_index_clean.csv


,transcript,clean_transcript
8118,showed the numerals twelve:thirty as the Vice-...,SHOWED THE NUMERALS TWELVETHIRTY AS THE VICE P...
9372,Oswald was seen in the vicinity of the southea...,OSWALD WAS SEEN IN THE VICINITY OF THE SOUTHEA...
4954,but it was proved that Palmer tried hard to ge...,BUT IT WAS PROVED THAT PALMER TRIED HARD TO GE...
6675,No amendment which any powerful economic inter...,NO AMENDMENT WHICH ANY POWERFUL ECONOMIC INTER...
4189,"Mr. Fasson, more and more ill at ease, would n...",MR FASSON MORE AND MORE ILL AT EASE WOULD NOT ...
4890,covered rather scantily with light sandy hair.,COVERED RATHER SCANTILY WITH LIGHT SANDY HAIR
6577,"In the case of Supreme Court justices, that pe...",IN THE CASE OF SUPREME COURT JUSTICES THAT PEN...
87,"were the leaders in this luckless change, thou...",WERE THE LEADERS IN THIS LUCKLESS CHANGE THOUG...
7783,"In September, the White House decided to permi...",IN SEPTEMBER THE WHITE HOUSE DECIDED TO PERMIT...
883,some prisons that had been ameliorated under t...,SOME PRISONS THAT HAD BEEN AMELIORATED UNDER T...


In [53]:
!pip install tqdm

In [54]:
import pandas as pd
import numpy as np
import librosa
import os
from pathlib import Path
from tqdm import tqdm

# Load the clean index
index_path = Path('/content/data/master_index_clean.csv')
df = pd.read_csv(index_path)

# Create a directory to store the NumPy arrays
features_dir = Path('/content/data/spectrograms')
features_dir.mkdir(parents=True, exist_ok=True)

# Paper parameters
target_sr = 16000
frame_length_ms = 200
stride_ms = 80

# Convert ms to samples based on target_sr
win_length = int(target_sr * (frame_length_ms / 1000.0))  # 3200 samples
hop_length = int(target_sr * (stride_ms / 1000.0))        # 1280 samples
n_fft = max(256, win_length) # n_fft must be >= win_length to avoid Librosa errors

def process_audio(row):
    audio_path = row['audio_filepath']
    # Create a unique filename for the .npy file using the dataset and original filename
    base_name = Path(audio_path).stem
    speaker = row['speaker_id']
    save_name = f"{row['dataset']}_{speaker}_{base_name}.npy"
    save_path = features_dir / save_name

    # Skip if we already processed this file (useful if the script gets interrupted)
    if save_path.exists():
        return str(save_path.resolve())

    try:
        # Load and resample to 16kHz
        y, sr = librosa.load(audio_path, sr=target_sr)

        # Generate the Spectrogram (Voicegram)
        stft = librosa.stft(y, n_fft=n_fft, hop_length=hop_length, win_length=win_length)

        # Convert to magnitude (absolute values) and convert to decibels
        spectrogram = librosa.amplitude_to_db(np.abs(stft), ref=np.max)

        # Save as a numpy array
        np.save(save_path, spectrogram)

        return str(save_path.resolve())

    except Exception as e:
        # Some audio files in large datasets are corrupted; we will catch them here
        # print(f"Error processing {audio_path}: {e}")
        return None

print("Starting the Spectrogram Factory...")
print(f"Processing {len(df)} files. This will take a while!")

# Apply processing with a progress bar
tqdm.pandas(desc="Extracting Voicegrams")
df['spectrogram_path'] = df.progress_apply(process_audio, axis=1)

# Drop any rows where the audio processing failed (corrupted wav files)
df_final = df.dropna(subset=['spectrogram_path']).copy()

# Save the final, ML-ready index!
final_index_path = Path('/content/data/master_index_ml_ready.csv')
df_final.to_csv(final_index_path, index=False)

print(f"\nFeature extraction complete! Saved {len(df_final)} spectrogram matrices.")

Starting the Spectrogram Factory...
Processing 13084 files. This will take a while!


Extracting Voicegrams: 100%|██████████| 13084/13084 [00:00<00:00, 13778.56it/s]



Feature extraction complete! Saved 13084 spectrogram matrices.


In [55]:
import pandas as pd
import torch
from pathlib import Path

class CharacterTokenizer:
    def __init__(self):
        # Define our allowed characters based on our cleaning script
        self.chars = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 '"

        # Define special tokens
        self.PAD_IDX = 0
        self.SOS_IDX = 1
        self.EOS_IDX = 2
        self.UNK_IDX = 3 # Unknown characters

        # Create mapping dictionaries
        self.char2idx = {char: idx + 4 for idx, char in enumerate(self.chars)}
        self.char2idx['<PAD>'] = self.PAD_IDX
        self.char2idx['<SOS>'] = self.SOS_IDX
        self.char2idx['<EOS>'] = self.EOS_IDX
        self.char2idx['<UNK>'] = self.UNK_IDX

        self.idx2char = {idx: char for char, idx in self.char2idx.items()}
        self.vocab_size = len(self.char2idx)

    def encode(self, text):
        """Converts a string to a list of integer tokens, wrapped in SOS and EOS."""
        tokens = [self.SOS_IDX]
        for char in text:
            tokens.append(self.char2idx.get(char, self.UNK_IDX))
        tokens.append(self.EOS_IDX)
        return tokens

    def decode(self, tokens):
        """Converts a list of integer tokens back to a string."""
        chars = []
        for token in tokens:
            if token in [self.PAD_IDX, self.SOS_IDX, self.EOS_IDX]:
                continue
            chars.append(self.idx2char.get(token, ''))
        return "".join(chars)

# Let's test it!
tokenizer = CharacterTokenizer()
sample_text = "HELLO WORLD"
encoded = tokenizer.encode(sample_text)
decoded = tokenizer.decode(encoded)

print(f"Original: {sample_text}")
print(f"Encoded:  {encoded}")
print(f"Decoded:  {decoded}")
print(f"Vocab Size: {tokenizer.vocab_size}")

Original: HELLO WORLD
Encoded:  [1, 11, 8, 15, 15, 18, 40, 26, 18, 21, 15, 7, 2]
Decoded:  HELLO WORLD
Vocab Size: 42


In [56]:
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class DysarthricSpeechDataset(Dataset):
    def __init__(self, csv_file, tokenizer):
        self.data_frame = pd.read_csv(csv_file)
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, idx):
        # 1. Load the spectrogram (X)
        spec_path = self.data_frame.iloc[idx]['spectrogram_path']
        spectrogram = np.load(spec_path)
        # Convert to PyTorch tensor and transpose if necessary to (Time, Frequency)
        spectrogram = torch.FloatTensor(spectrogram).T

        # 2. Load and tokenize the text (Y)
        transcript = self.data_frame.iloc[idx]['clean_transcript']
        tokens = self.tokenizer.encode(transcript)
        tokens = torch.LongTensor(tokens)

        return spectrogram, tokens

def collate_fn(batch):
    """Pads variable-length sequences to the max length in the batch."""
    specs, tokens = zip(*batch)

    # Store original lengths before padding (useful for the Transformer later)
    spec_lengths = torch.LongTensor([s.size(0) for s in specs])
    token_lengths = torch.LongTensor([t.size(0) for t in tokens])

    # Pad the spectrograms with zeros
    specs_padded = pad_sequence(specs, batch_first=True, padding_value=0.0)

    # Pad the text tokens with the PAD_IDX
    tokens_padded = pad_sequence(tokens, batch_first=True, padding_value=0) # 0 is PAD_IDX

    return specs_padded, tokens_padded, spec_lengths, token_lengths

# --- Setup the DataLoader ---
index_path = '/content/data/master_index_ml_ready.csv'
dataset = DysarthricSpeechDataset(index_path, tokenizer)

# Create the loader (batch size of 16 for testing)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)

# Let's pull one batch to verify it works
for batch_specs, batch_tokens, spec_lens, token_lens in dataloader:
    print(f"Batch Spectrograms Shape: {batch_specs.shape}") # Should be (16, Max_Time, Freq_Bins)
    print(f"Batch Tokens Shape:       {batch_tokens.shape}") # Should be (16, Max_Text_Length)
    break # We only need to see one batch to know it works!

Batch Spectrograms Shape: torch.Size([16, 119, 1601])
Batch Tokens Shape:       torch.Size([16, 164])


In [57]:
import torch
import torch.nn as nn

class DepthwiseSeparableConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, padding):
        super().__init__()
        # 1. Depthwise Convolution (groups = in_channels)
        self.depthwise = nn.Conv1d(
            in_channels, in_channels, kernel_size=kernel_size,
            padding=padding, groups=in_channels
        )
        # 2. Pointwise Convolution (1x1 kernel)
        self.pointwise = nn.Conv1d(
            in_channels, out_channels, kernel_size=1
        )

    def forward(self, x):
        # x expected shape: (Batch, Channels, Time)
        x = self.depthwise(x)
        x = self.pointwise(x)
        return x

In [58]:
class Transformer2EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        # First Attention Block
        self.attn1 = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)

        # Two Depthwise Separable Convolutions replacing the standard FFN
        self.dw_conv1 = DepthwiseSeparableConv1d(d_model, d_model, kernel_size=3, padding=1)
        self.dw_conv2 = DepthwiseSeparableConv1d(d_model, d_model, kernel_size=3, padding=1)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout)

        # Second Attention Block
        self.attn2 = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, dropout=dropout, batch_first=True)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, src_key_padding_mask=None):
        # x expected shape: (Batch, Time, d_model)

        # --- First Attention Block ---
        # Residual connection 1
        attn1_out, _ = self.attn1(x, x, x, key_padding_mask=src_key_padding_mask)
        x = self.norm1(x + self.dropout1(attn1_out))

        # --- Depthwise Separable Block ---
        # Conv1d expects (Batch, Channels, Time), but Transformer uses (Batch, Time, Channels)
        # So we transpose, convolve, and transpose back
        res2 = x # Save residual
        x_conv = x.transpose(1, 2)
        x_conv = torch.relu(self.dw_conv1(x_conv))
        x_conv = torch.relu(self.dw_conv2(x_conv))
        x_conv = x_conv.transpose(1, 2)

        x = self.norm2(res2 + self.dropout2(x_conv))

        # --- Second Attention Block ---
        res3 = x
        attn2_out, _ = self.attn2(x, x, x, key_padding_mask=src_key_padding_mask)
        x = self.norm3(res3 + self.dropout3(attn2_out))

        return x

In [59]:
# Test the Encoder Block
batch_size = 16
seq_length = 120
d_model = 256 # Standard hidden dimension for the model
n_heads = 8

# Create a dummy input mimicking the output of the down-sampling layers
dummy_input = torch.rand(batch_size, seq_length, d_model)

encoder = Transformer2EncoderBlock(d_model=d_model, n_heads=n_heads)
output = encoder(dummy_input)

print(f"Encoder Input Shape:  {dummy_input.shape}")
print(f"Encoder Output Shape: {output.shape}")

Encoder Input Shape:  torch.Size([16, 120, 256])
Encoder Output Shape: torch.Size([16, 120, 256])


In [60]:
import torch
import torch.nn as nn
import math

# ==========================================
# 1. Positional Encoding
# ==========================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Compute standard sinusoidal positional encodings
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0)) # Registers as state, not a parameter to be trained

    def forward(self, x):
        # x shape: (Batch, Time, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

# ==========================================
# 2. The Convolutional Frontend (Down-sampler)
# ==========================================
class SpeechFrontend(nn.Module):
    def __init__(self, input_freq_bins, d_model, dropout=0.1):
        super().__init__()
        # Paper spec: 3 layers, 64 filters, kernel 11, stride 2
        # We map the final 64 filters to d_model so it fits into the Transformer
        self.conv1 = nn.Conv1d(input_freq_bins, 64, kernel_size=11, stride=2, padding=5)
        self.conv2 = nn.Conv1d(64, 64, kernel_size=11, stride=2, padding=5)
        self.conv3 = nn.Conv1d(64, d_model, kernel_size=11, stride=2, padding=5)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Input x is (Batch, Time, Freq_Bins)
        # PyTorch Conv1d expects (Batch, Channels, Time) - so we transpose Time and Freq
        x = x.transpose(1, 2)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.relu(self.conv3(x))
        x = self.dropout(x)
        # Transpose back to (Batch, Time, d_model) for the Transformer
        return x.transpose(1, 2)

# ==========================================
# 3. The Full Sequence-to-Sequence Model
# ==========================================
class DysarthricSpeechTransformer(nn.Module):
    def __init__(self, input_freq_bins, vocab_size, d_model=256, n_heads=8, num_encoder_layers=5, num_decoder_layers=3):
        super().__init__()
        self.d_model = d_model

        # 1. Frontend & Positional Encoding for Audio
        self.frontend = SpeechFrontend(input_freq_bins, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        # 2. Text Embedding & Positional Encoding for Target Tokens
        self.tgt_emb = nn.Embedding(vocab_size, d_model)
        self.pos_decoder = PositionalEncoding(d_model)

        # 3. Custom Encoders (Transformer 2 Architecture)
        # Assuming you still have Transformer2EncoderBlock defined in your previous cell!
        self.encoders = nn.ModuleList([
            Transformer2EncoderBlock(d_model, n_heads) for _ in range(num_encoder_layers)
        ])

        # 4. Standard PyTorch Decoder
        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=n_heads, batch_first=True)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_decoder_layers)

        # 5. Output Layer
        self.out = nn.Linear(d_model, vocab_size)

    def forward(self, src, tgt, src_padding_mask=None, tgt_padding_mask=None, tgt_mask=None):
        # src: Spectrograms (Batch, Time, Freq)
        # tgt: Text Tokens (Batch, Target_Seq_Len)

        # --- Audio Encoding ---
        memory = self.frontend(src)
        memory = self.pos_encoder(memory)

        # Since the frontend conv layers shrunk the sequence by a factor of 8 (stride 2^3),
        # we would technically need to shrink the src_padding_mask here if we are using it.
        # For simplicity in this test, we'll let the model learn the padded silence.

        for encoder in self.encoders:
            memory = encoder(memory, src_key_padding_mask=None)

        # --- Text Decoding ---
        tgt_embed = self.tgt_emb(tgt) * math.sqrt(self.d_model)
        tgt_embed = self.pos_decoder(tgt_embed)

        # Pass through native decoder
        output = self.decoder(
            tgt=tgt_embed,
            memory=memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_padding_mask
        )

        # Get character probabilities
        return self.out(output)

In [61]:
# Based on your previous output shapes:
batch_size = 16
freq_bins = 1601
vocab_size = tokenizer.vocab_size

# Initialize the beast
model = DysarthricSpeechTransformer(
    input_freq_bins=freq_bins,
    vocab_size=vocab_size,
    d_model=256,
    n_heads=8,
    num_encoder_layers=5,
    num_decoder_layers=3
)

# Dummy data based on your dataloader shapes
dummy_spectrogram = torch.rand(batch_size, 120, freq_bins) # (Batch, Time, Freq)
dummy_text_tokens = torch.randint(0, vocab_size, (batch_size, 142)) # (Batch, Text_Len)

# We need a causal mask for the target so the model can't "look ahead" at future characters
tgt_seq_len = dummy_text_tokens.size(1)
causal_mask = nn.Transformer.generate_square_subsequent_mask(tgt_seq_len)

# Push it through the model!
final_output = model(dummy_spectrogram, dummy_text_tokens, tgt_mask=causal_mask)

print(f"Final Model Output Shape: {final_output.shape}")
# We want this to be [16, 142, vocab_size] (Batch, Text_Length, Character_Probabilities)

Final Model Output Shape: torch.Size([16, 142, 42])


In [62]:
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast

# Move model to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# We must ignore the padding token in the loss calculation!
# Assuming PAD_IDX was 0 in your CharacterTokenizer
criterion = nn.CrossEntropyLoss(ignore_index=0)

# AdamW is generally preferred over standard Adam for Transformers
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

# Gradient Scaler for Mixed Precision
scaler = GradScaler()

print(f"Model moved to {device}. Engine ready.")

Model moved to cuda. Engine ready.


/tmp/ipykernel_772/198814890.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [63]:
!pip install wandb

In [64]:
!wandb login

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: manhas-mridul05 (manhas-mridul05-thapar-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
import time
import wandb
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
import torch.nn as nn
import torch

# Initialize W&B
wandb.init(
    project="dysarthric-speech-transformer",
    name="ljspeech-base-run-01",
    config={
        "learning_rate": 3e-5, # LOWERED from 1e-4
        "epochs": 50,
        "batch_size": 16, # Assuming your DataLoader batch size
        "accumulation_steps": 4,
        "architecture": "Transformer-2"
    }
)

# Move model to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Tell W&B to watch the model's gradients and weights
wandb.watch(model, log="all", log_freq=100)

criterion = nn.CrossEntropyLoss(ignore_index=0)
# Updated Learning Rate
optimizer = optim.AdamW(model.parameters(), lr=3e-5, weight_decay=1e-5)
scaler = GradScaler()

epochs = wandb.config.epochs
accumulation_steps = wandb.config.accumulation_steps
model.train()

print("Initiating W&B Base Model Training...")
global_step = 0

for epoch in range(epochs):
    epoch_loss = 0
    start_time = time.time()
    optimizer.zero_grad()

    for batch_idx, (specs, tokens, spec_lens, token_lens) in enumerate(dataloader):
        specs = specs.to(device)
        tokens = tokens.to(device)

        tgt_input = tokens[:, :-1]
        tgt_expected = tokens[:, 1:]

        tgt_seq_len = tgt_input.size(1)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(tgt_seq_len).to(device)

        # Mixed Precision Forward Pass
        with autocast():
            output = model(specs, tgt_input, tgt_mask=causal_mask)

            output_dim = output.shape[-1]
            output = output.contiguous().view(-1, output_dim)
            tgt_expected = tgt_expected.contiguous().view(-1)

            # CRITICAL FIX: Cast output back to float32 before calculating loss to prevent NaN
            loss = criterion(output.float(), tgt_expected)
            loss = loss / accumulation_steps

        scaler.scale(loss).backward()

        if (batch_idx + 1) % accumulation_steps == 0:
            scaler.unscale_(optimizer)
            # Tightened gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

            global_step += 1

            # --- W&B: Log batch metrics ---
            wandb.log({"Batch Loss": loss.item() * accumulation_steps}, step=global_step)

        epoch_loss += loss.item() * accumulation_steps

        if batch_idx % 100 == 0:
            current_loss = loss.item() * accumulation_steps
            print(f"Epoch {epoch+1} | Batch {batch_idx}/{len(dataloader)} | Loss: {current_loss:.4f}")

    epoch_time = time.time() - start_time
    avg_loss = epoch_loss / len(dataloader)

    # --- W&B: Log epoch metrics ---
    wandb.log({"Epoch Average Loss": avg_loss, "Epoch": epoch + 1})

    print(f"=== Epoch {epoch+1} Complete | Avg Loss: {avg_loss:.4f} | Time: {epoch_time:.2f}s ===")
    torch.save(model.state_dict(), f"/content/data/transformer2_ljspeech_epoch{epoch+1}.pt")

wandb.finish()

Epoch,▁
+2,...
Batch Loss,nan
Epoch,1
Epoch Average Loss,nan


/tmp/ipykernel_772/200231033.py:31: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_772/200231033.py:56: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Initiating W&B Base Model Training...
Epoch 1 | Batch 0/818 | Loss: 3.9283
Epoch 1 | Batch 100/818 | Loss: 2.7622
Epoch 1 | Batch 200/818 | Loss: 2.6245
Epoch 1 | Batch 300/818 | Loss: 2.4500
Epoch 1 | Batch 400/818 | Loss: 2.4538
Epoch 1 | Batch 500/818 | Loss: 2.4097
Epoch 1 | Batch 600/818 | Loss: 2.3619
Epoch 1 | Batch 700/818 | Loss: 2.3665
Epoch 1 | Batch 800/818 | Loss: 2.4868
=== Epoch 1 Complete | Avg Loss: 2.5389 | Time: 98.88s ===
Epoch 2 | Batch 0/818 | Loss: 2.3644
Epoch 2 | Batch 100/818 | Loss: 2.4212
Epoch 2 | Batch 200/818 | Loss: 2.3014
Epoch 2 | Batch 300/818 | Loss: 2.4040
Epoch 2 | Batch 400/818 | Loss: 2.4032
Epoch 2 | Batch 500/818 | Loss: 2.3899
Epoch 2 | Batch 600/818 | Loss: 2.3626
Epoch 2 | Batch 700/818 | Loss: 2.3451
Epoch 2 | Batch 800/818 | Loss: 2.3676
=== Epoch 2 Complete | Avg Loss: 2.3659 | Time: 85.91s ===
Epoch 3 | Batch 0/818 | Loss: 2.4175
Epoch 3 | Batch 100/818 | Loss: 2.3238
Epoch 3 | Batch 200/818 | Loss: 2.3033
Epoch 3 | Batch 300/818 | Loss: